# League Of Legends 15 Minute Match Predictor

**Name(s)**: Don Le

**Website Link**: [League Of Legends Match Predictor Website](https://donle727.github.io/League-Of-legends-Tier-1-League-Match-Predictor/)

In [116]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.express as px
pd.options.plotting.backend = 'plotly'

from dsc80_utils import * # Feel free to uncomment and use this.

np.random.seed(42)

## Introduction

For this project, we are analyizing competitive League of Legends esports sourced from [Oracle's Elixer data](https://oracleselixir.com/tools/downloads). This dataset provides post-match statistics from official professional leagues from 2014 to 2026. 

Despite the many professional leagues that are provided with the data, we will focus on the high-skill and high-staked, namely, Tier-1 leagues (CBLOL, LCK, LCP, LCS, LEC, and LPL). By narrowing our scope to these intense and unforgiving environments, we can better understand optimal gameplay and precise strategy with minimal noise often found in lower-tier professional play.

During our initial exploration of the data, we came up with some intersting questions regarding what actually drives a professional team to victory:
* "Is there any difference in kills between professional leagues?"
    * What's the difference in intensity and aggression between Tier-1 leagues and the rest of the competitive pool?
* "How can first objective (like dragon) can affect winrate?"
    * Does priority of objectives allow a team to leverage victory against their opponent? 
* "How much does early-game snowballing effect win rate?"
    * How much does early-game gold advantage help a team to win?

## Data Cleaning and Exploratory Data Analysis

### DataFrame
* 165 Variables; [Variable Definitions Here](https://oracleselixir.com/definitions)
* 376452 Observations; 2 observations per game
* Has many missing data

In [117]:
# DataFrame Initialization (Raw Data)

csv_files = ['2023_LoL_esports_match_data_from_OraclesElixir.csv',
             '2024_LoL_esports_match_data_from_OraclesElixir.csv', 
             '2025_LoL_esports_match_data_from_OraclesElixir.csv']

data_folder = Path('data')

df_main = pd.concat([pd.read_csv(data_folder / f, low_memory=False) for f in csv_files], ignore_index=True) # Concatenate LoL Pro games from 2023-2026

df_main

,gameid,datacompleteness,url,league,...,deathsat25,opp_killsat25,opp_assistsat25,opp_deathsat25
0,ESPORTSTMNT06_2753012,complete,NaN,LFL2,...,0.0,0.0,1.0,0.0
1,ESPORTSTMNT06_2753012,complete,NaN,LFL2,...,1.0,0.0,1.0,0.0
2,ESPORTSTMNT06_2753012,complete,NaN,LFL2,...,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...
376449,LOLTMNT03_332179,complete,NaN,DCup,...,1.0,1.0,1.0,4.0
376450,LOLTMNT03_332179,complete,NaN,DCup,...,13.0,13.0,34.0,8.0
376451,LOLTMNT03_332179,complete,NaN,DCup,...,8.0,8.0,18.0,13.0


### Leagues

In [118]:
print(list(df_main['league'].unique()))

['LFL2', 'DDH', 'EL', 'LPL', 'GL', 'LCKC', 'NEXO', 'UL', 'LVP SL', 'LCK', 'LFL', 'PRM', 'LMF', 'SL (LATAM)', 'VL', 'CBLOL', 'LEC', 'NACL', 'LCO', 'CBLOLA', 'LHE', 'NLC', 'GLL', 'ESLOL', 'LLA', 'EBL', 'TCL', 'PGN', 'LPLOL', 'LCS', 'HM', 'LJL', 'HC', 'AL', 'PCS', 'LDL', 'VCS', 'EM', 'MSI', 'LAS', 'LRN', 'LRS', 'EPL', 'LJLA', 'CT', 'WLDs', 'PCL', 'ASCI', 'CDF', 'IC', 'USP', 'DCup', 'TSC', 'LIT', 'IGNIS', 'EWC', 'AC', 'PRMP', 'HW', 'NLC Aurora Open', 'EBLPA', 'GLLPA', 'KeSPA', 'LCP', 'HLL', 'LTA S', 'LTA N', 'RL', 'CD', 'ROL', 'LTA', 'FST', 'Asia Master', 'ASI']


### Columns

In [119]:
print(list(df_main.columns))

['gameid', 'datacompleteness', 'url', 'league', 'year', 'split', 'playoffs', 'date', 'game', 'patch', 'participantid', 'side', 'position', 'playername', 'playerid', 'teamname', 'teamid', 'firstPick', 'champion', 'ban1', 'ban2', 'ban3', 'ban4', 'ban5', 'pick1', 'pick2', 'pick3', 'pick4', 'pick5', 'gamelength', 'result', 'kills', 'deaths', 'assists', 'teamkills', 'teamdeaths', 'doublekills', 'triplekills', 'quadrakills', 'pentakills', 'firstblood', 'firstbloodkill', 'firstbloodassist', 'firstbloodvictim', 'team kpm', 'ckpm', 'firstdragon', 'dragons', 'opp_dragons', 'elementaldrakes', 'opp_elementaldrakes', 'infernals', 'mountains', 'clouds', 'oceans', 'chemtechs', 'hextechs', 'dragons (type unknown)', 'elders', 'opp_elders', 'firstherald', 'heralds', 'opp_heralds', 'void_grubs', 'opp_void_grubs', 'firstbaron', 'barons', 'opp_barons', 'atakhans', 'opp_atakhans', 'firsttower', 'towers', 'opp_towers', 'firstmidtower', 'firsttothreetowers', 'turretplates', 'opp_turretplates', 'inhibitors', '

### Clean



In [120]:
keep_cols = ['gameid','league', 'side', 'position', 'result', 'firstblood', 'firstdragon', 'golddiffat15', 'killsat15','opp_killsat15', 'ckpm']
tier_one_leagues = ['LCK', 'LEC', 'LCS', 'CBLOL', 'LCP', 'LPL']

df_clean = (
    df_main[keep_cols]
    .loc[df_main['league'].isin(tier_one_leagues)]
    .loc[df_main['position'] == 'team']
    .assign(
        result=lambda x: x['result'].astype(int),
        firstblood=lambda x: x['firstblood'].astype('Int64'),
        firstdragon=lambda x: x['firstdragon'].astype('Int64')
    )
    .reset_index(drop=True).dropna() # We will drop NA for consistency and accuracy.
)

display(df_clean.head())

,gameid,league,side,position,...,golddiffat15,killsat15,opp_killsat15,ckpm
46,ESPORTSTMNT04_2659018,LCK,Blue,team,...,3176.0,5.0,1.0,0.80
47,ESPORTSTMNT04_2659018,LCK,Red,team,...,-3176.0,1.0,5.0,0.80
48,ESPORTSTMNT04_2661035,LCK,Blue,team,...,1287.0,3.0,0.0,0.58
49,ESPORTSTMNT04_2661035,LCK,Red,team,...,-1287.0,0.0,3.0,0.58
54,ESPORTSTMNT04_2660040,LCK,Blue,team,...,905.0,2.0,4.0,0.83


In [121]:
agg_stats = (
    df_clean.groupby('league')
    .agg(
        avg_ckpm=('ckpm', 'mean'),
        avg_kills_at_15=('killsat15', 'mean'),
        first_blood_rate=('firstblood', 'mean'), # Since it's 1/0, the mean is the percentage!
        first_dragon=('firstdragon','mean')
    )
    .round(3)
    .sort_values(by='avg_ckpm', ascending=False)
)

display(agg_stats)

,avg_ckpm,avg_kills_at_15,first_blood_rate,first_dragon
league,,,,
LCP,0.87,3.41,0.5,0.5
LEC,0.85,3.76,0.5,0.5
CBLOL,0.83,2.72,0.5,0.5
LCS,0.81,3.34,0.5,0.5
LCK,0.80,3.01,0.5,0.5


### Univariate Analysis #1 The distribution of Gold Difference at 15 minutes (All Leagues *NOT Inlcuding LPL, LDL, DCup, ACSI)
LPL, LDL, DCup, ACSI's have restricted API's which do not measure Gold difference at 15 minutes. It is important to remember some games in MSI, WLDs, LLA, LCK, LCO, ESLOL have occasional missing data for gold difference at 15 minutes. Our accuracy and representation of the population is therefore not fully complete/reflected.

In [122]:
# Plotting the distribution of Gold Difference at 15 minutes (All Leagues *NOT Inlcuding LPL, LDL, DCup, ACSI)
fig = px.histogram(
    df_main, 
    x="golddiffat15", 
    nbins=50,
    title="Distribution of Gold Difference at 15 Minutes (All Leagues but LPL, LDL, DCup, ACSI)",
    labels={"golddiffat15": "Gold Difference", "count": "Number of Games"},
    color_discrete_sequence=['#636EFA']
)

# Add a vertical line at 0 to show the perfect center
fig.add_vline(x=0, line_width=3, line_dash="solid", line_color="red", annotation_text="Dead Even (0g)")

fig.update_layout(width=800, height=500, xaxis=dict(range=[-6000, 6000]))
fig.show()
fig.write_html('assets/dist_of_gold_all_leagues_histogram.html', include_plotlyjs='cdn')

### Univariate Analysis #2 (Tier-1 Leagues *NOT including LPL)

In [123]:
# Plotting the distribution of Gold Difference at 15 minutes (Tier-1)
fig = px.histogram(
    df_clean, 
    x="golddiffat15", 
    nbins=50,
    title="Distribution of Gold Difference at 15 Minutes (Tier-1 Leagues)",
    labels={"golddiffat15": "Gold Difference", "count": "Number of Games"},
    color_discrete_sequence=['#636EFA']
)

# Add a vertical line at 0 to show the perfect center
fig.add_vline(x=0, line_width=5 , line_dash="solid", line_color="red", annotation_text="Dead Even (0g)")

fig.update_layout(width=800, height=500)
fig.show()
fig.write_html('assets/dist_of_gold_tier_one_histogram.html', include_plotlyjs='cdn')


### Bivariate Analysis #1: The Distribution of CKPM by League
There were no missing data for CPKM.

Tier one leagues have surprisingly low combined kills per minute relative to the other leagues.

In [124]:
# Get df_main with all other leagues
df_eda = df_main[df_main['position'] == 'team'].copy()


df_eda['league_group'] = df_eda['league'].apply(
    lambda x: 'Tier One' if x in tier_one_leagues else 'Other Leagues'
)

# Group by league
league_order = df_eda.groupby("league")['ckpm'].mean().sort_values(ascending=False).index

fig = px.box(
    df_eda, 
    x="league",
    y="ckpm",
    color="league_group",
    category_orders={'league': league_order},
    color_discrete_map={'Tier One': '#636EFA', 'Other Leagues': '#EF553B'},
    title="Distribution of CKPM by League",
    points="outliers",
)

fig.update_layout(
    width=1200, 
    height=600,
    title_x=0.5 
)

fig.write_html('assets/ckpm_dist_by_league_box_plot.html', include_plotlyjs='cdn')
fig.show()

### Bivariate Analysis #2: The Relationship between Kill Lead and Gold Lead at 15 Minutes by Map Side

In our second bivariate analysis, we examined the relationship between kill difference (calculated using *killsat15* and *opp_killsat15*) and gold difference at the 15-minute mark. We separated and categorized these relationships with the *side* that a team is on. By measuring the kill difference against gold different at the 15 minute mark between blue and red side, we are able to anaylze the correlation of economic performance based on map side. By faceting the analysis side by side, we can observe that the regression lines and scatter plots remain nearly identical, suggesting that at the 15 minute mark of top professional play, map side does not grant gold and kill advantage.

It is important to note that in certain games and regions, most notably the LPL, are missing minute-by-minute data. Therefore for the integrity and accuracy of this analysis, we decided to drop these observations entirely, focusing on the LCK, LEC, LCS, CBLOL, and LCP and games that have sufficient data for this analysis.


In [125]:
# 1. Create the new column: Kill Difference at 15 minutes
df_cleaned_na = df_clean.dropna(subset=['killsat15','opp_killsat15']).copy()
df_cleaned_na['killdiffat15'] = df_cleaned_na['killsat15'] - df_cleaned_na['opp_killsat15']


# 3. Map results for the discrete legend
df_cleaned_na['Match Result'] = df_cleaned_na['result'].map({0: 'Loss', 1: 'Win'})


fig_facets = px.scatter(
    df_cleaned_na, 
    x="killdiffat15", 
    y="golddiffat15",
    color="Match Result", 
    facet_col="side",
    trendline="ols",
    trendline_scope="overall",
    title="<b>Kill Lead vs. Gold Advantage: Is there a Side Bias?</b>",
    labels={
        "killdiffat15": "Kill Difference (15m)", 
        "golddiffat15": "Gold Difference (15m)",
        "Match Result": "Final Outcome",
        "side": "Map Side"
    },
    color_discrete_map={'Loss': "#FF2600", 'Win': "#0011FF"},
    opacity=0.4
)

# High-end layout adjustments
fig_facets.update_layout(
    width=1000, 
    height=600,
    title_x=0.5, # Center the title
    font=dict(family="Arial, sans-serif", size=12)
)

# Clean up the subplot headers (removes "side=Blue" and makes it just "Blue")
fig_facets.for_each_annotation(lambda a: a.update(text=f"<b>{a.text.split('=')[-1]} Side</b>"))

fig_facets.write_html('assets/kill_lead_vs_gold_lead_scatter.html', include_plotlyjs='cdn')
fig_facets.show()

## Assessment of Missingness

Upon further examination of our dataset, LPL unfortunately did not measure data on kills and gold difference at 15 minute. Therefore, our research will have to exclude LPL entirely. We assume this to be Missing by Design because of how Chinese matches are hosted differently from those hosted by Riot themselves (foot note 1). We noticed  there were also other some observations that seemed to be missing randomly from specific variables such as 'firstdragon'.

Therefore, we completed permutation tests to analyze the dependency of missingness. Our threshold of significance will be a p-value of 0.05.
- Null Hypothesis: The missingness of *firstdragon*  does **not** depend on the *league*.
- Alternative Hypothesis: The missingness of *firstdragon* **does** depend on the *league*.

Our results from our permutation test begun with an served TVD of 0.0588 and our p-value was ~0.0000. Because our p-value was far under our threshold (0.05), we therefore reject the null hypothesis. The missingness of *firstdragon* is dependent on the *league* the game was played in. This is an example of Missing At Random (MAR).


Foot Note:
1. [Source](https://github.com/FloPrm/lol_analytics?tab=readme-ov-file#tv-esports-data)

In [126]:
missing_fifteen_stats = df_main[df_main['league'] == 'LPL'][['killsat15','golddiffat15']]
missing_fifteen_stats
print(f'Missing Observations: {missing_fifteen_stats['golddiffat15'].isna().sum()}')
print(f'LPL League Games: {missing_fifteen_stats.shape[0]}')


Missing Observations: 27324
LPL League Games: 27324


In [127]:
# 1. See total missing values for firstdragon
print(f"Total missing firstdragon: {df_main['firstdragon'].isna().sum()}")

# 2. See which leagues have the most missing firstdragon data
missing_dragons = df_main[df_main['firstdragon'].isna()]
print(missing_dragons['league'].value_counts())

Total missing firstdragon: 316830
league
LPL      22776
LDL      17052
NACL     16310
         ...  
AC         350
FST        350
EBLPA      250
Name: count, Length: 74, dtype: int64


In [128]:
# See which leagues have the most missing golddiffat15 data
missing_by_league = (
    df_main[df_main['golddiffat15'].isna()]
    ['league'].value_counts()
)

print("Leagues with the most missing 15-minute stats:")
print(missing_by_league.head(10))
print(list(missing_by_league.index))

Leagues with the most missing 15-minute stats:
league
LPL      27324
LDL      17052
MSI        936
         ...  
LCK         12
ESLOL       12
LCO         12
Name: count, Length: 10, dtype: int64
['LPL', 'LDL', 'MSI', 'ASCI', 'DCup', 'WLDs', 'LLA', 'LCK', 'ESLOL', 'LCO']


In [ ]:
target_col = 'firstdragon'
dependency_col = 'league'
# dependency_col = 'side'    # Swap to this for your MCAR test (Test 2)


missing_df = df_main[[target_col, dependency_col]].copy()


missing_df['is_missing'] = missing_df[target_col].isna()


def calculate_tvd(df, missing_col, dep_col):

    dist = df.pivot_table(
        index=dep_col, 
        columns=missing_col, 
        aggfunc='size', 
        fill_value=0
    )

    dist = dist / dist.sum() 
    
    tvd = dist.diff(axis=1).iloc[:, -1].abs().sum() / 2
    return tvd

observed_tvd = calculate_tvd(missing_df, 'is_missing', dependency_col)
print(f"Observed TVD: {observed_tvd:.4f}")

# Permutation Test
np.random.seed(42)
n_repetitions = 500 
tvds = []
for _ in range(n_repetitions):
    shuffled_col = np.random.permutation(missing_df[dependency_col])
    
    temp_df = missing_df.assign(shuffled=shuffled_col)
    simulated_tvd = calculate_tvd(temp_df, 'is_missing', 'shuffled')
    tvds.append(simulated_tvd)

# P-value
p_value = np.count_nonzero(np.array(tvds) >= observed_tvd) / n_repetitions
print(f"P-Value: {p_value:.4f}")

Observed TVD: 0.0588
P-Value: 0.0000


## Hypothesis Testing

1. Null Hypothesis: The Mean Combined Kills per Minute (CKPM) in Tier 1 Leagues is the same as the mean CKPM of other Leagues.
2. Alternative Hypothesis: The CKPM of Tier 1 Leagues is ***less*** than the mean CKPM of other leagues. 

In [ ]:
# 1. We assume 'df' is your full combined dataset from Step 1
# Get game-level data for ALL leagues
hyp_df = df_main[df_main['position'] == 'team'].drop_duplicates('gameid').copy()

# 2. Define Tier-One and create a True/False column
tier_one = ['CBLOL','LCK','LCP','LSC','LEC','LPL']
hyp_df['is_tier_one'] = hyp_df['league'].isin(tier_one)

# 3. Split into the two groups
tier_1_action = hyp_df[hyp_df['is_tier_one'] == True]['ckpm']
other_action = hyp_df[hyp_df['is_tier_one'] == False]['ckpm']

# 4. Calculate the Observed Difference (Tier 1 - Others)
# We expect this to be a NEGATIVE number if Tier 1 is less bloody
observed_diff = tier_1_action.mean() - other_action.mean()

print(f"Tier-One Mean CKPM: {tier_1_action.mean():.4f}")
print(f"Other Leagues Mean CKPM: {other_action.mean():.4f}")
print(f"Observed Test Statistic (Difference in Means): {observed_diff:.4f}")

# 1. Setup the data arrays
ckpm_values = hyp_df['ckpm'].values
n_tier_one = hyp_df['is_tier_one'].sum() # Total number of Tier-One games

n_permutations = 10000
differences = []

# 2. The Permutation Loop
np.random.seed(42)
for _ in range(n_permutations):
    # Shuffle all the CKPM values
    shuffled_ckpm = np.random.permutation(ckpm_values)
    
    # Split into fake groups based on the original group sizes
    fake_tier_1 = shuffled_ckpm[:n_tier_one]
    fake_others = shuffled_ckpm[n_tier_one:]
    
    # Calculate the fake difference in means
    fake_diff = fake_tier_1.mean() - fake_others.mean()
    differences.append(fake_diff)

differences_array = np.array(differences)

# 3. Calculate the P-value
# We use <= because we are testing if Tier 1 is significantly LOWER
p_value = np.mean(differences_array <= observed_diff)

print(f"Observed Difference: {observed_diff:.4f}")
print(f"P-value: {p_value}")

# 4. Visualize the Results
dist_df = pd.DataFrame({'diffs': differences})

fig = px.histogram(
    dist_df, 
    x='diffs', 
    nbins=50,
    title=f'Permutation Test: Tier-One vs Other Leagues<br><sup>P-value: {p_value}</sup>',
    labels={'diffs': 'Difference in Mean CKPM (Tier-One - Others)'},
    template='plotly_white'
)

# Add the red line for our actual observed difference
fig.add_vline(
    x=observed_diff, 
    line_dash="dash", 
    line_color="red", 
    annotation_text=f"Observed: {observed_diff:.4f}", 
    annotation_position="top left" # Positioned left because it's a negative value
)

fig.show()
fig.write_html('assets/permutation_test_histogram.html', include_plotlyjs='cdn')

Tier-One Mean CKPM: 0.8393
Other Leagues Mean CKPM: 1.0041
Observed Test Statistic (Difference in Means): -0.1648
Observed Difference: -0.1648
P-value: 0.0


## Framing a Prediction Problem

Can we predict a win or loss of a pro League of Legends match using the information available at exactly the 15-minute mark?

## Baseline Model (Logistic Regression)

In [131]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# 1. Prepare Data (Only Gold this time)
model_leagues = ['LCK', 'LEC', 'LCS', 'CBLOL', 'LCP'] 
model_df = df_clean[df_clean['league'].isin(model_leagues)].dropna(subset=['golddiffat15', 'result'])

X = model_df[['golddiffat15']]
y = model_df['result']

# 2. Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Pipeline
baseline_model = Pipeline([
    ('scaler', StandardScaler()), # Standardize the data
    ('classifier', LogisticRegression()) # Logistic Regression
])

# 4. Train & Score
baseline_model.fit(X_train, y_train)

print(f"Train Accuracy: {baseline_model.score(X_train, y_train):.4f}")
print(f"Test Accuracy: {baseline_model.score(X_test, y_test):.4f}")

Train Accuracy: 0.7155
Test Accuracy: 0.7224


In [132]:
gold_range = np.linspace(-5000, 5000, 500).reshape(-1, 1)

X_plot = pd.DataFrame(gold_range, columns=['golddiffat15'])

probs = baseline_model.predict_proba(X_plot)[:, 1]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=gold_range.flatten(), 
    y=probs, 
    name='Win Probability', 
    line=dict(color='#636EFA', width=3)
))

 # 50/50 threshold line
fig.add_shape(
    type="line", line_color="red", line_dash="dash",
    x0=-5000, x1=5000, y0=0.5, y1=0.5
)

fig.update_layout(
    title="Win Probability vs. Gold Differential",
    xaxis_title="Gold Difference at 15 Minutes",
    yaxis_title="Probability of Winning",
    template="plotly_white",
    width=900, height=500
)

fig.show()
fig.write_html('assets/baseline_logistic_regression.html', include_plotlyjs='cdn')

In [133]:
from sklearn.metrics import confusion_matrix

# 1. Get predictions on the test set
y_pred = baseline_model.predict(X_test)

# 2. Calculate the matrix
cm = confusion_matrix(y_test, y_pred)

# 3. Plot it as a heatmap
fig_cm = px.imshow(
    cm, 
    text_auto=True, 
    aspect="auto",
    labels=dict(x="Predicted Result", y="Actual Result"),
    x=['Loss (0)', 'Win (1)'],
    y=['Loss (0)', 'Win (1)'],
    title="Baseline Model: Confusion Matrix",
    color_continuous_scale='Blues'
)

fig_cm.show()
fig_cm.write_html('assets/base_confusion_matrix.html', include_plotlyjs='cdn')

## Final Model (Random Forest With 5-fold Cross-Validation)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV

stats = ['golddiffat15', 'killsat15', 'firstblood', 'firstdragon']
df_f = df_clean[df_clean['league'].isin(['LCK', 'LEC', 'LCS'])].dropna(subset=stats + ['result'])

X_train, X_test, y_train, y_test = train_test_split(df_f[stats], df_f['result'], test_size=0.2, random_state=42)

final_pl = Pipeline([
    ('ct', ColumnTransformer([('num', StandardScaler(), ['golddiffat15', 'killsat15'])], remainder='passthrough')),
    ('rf', RandomForestClassifier(max_depth=4, random_state=42))
])

param_grid = {
    'rf__max_depth': [3, 5, 7, 10], # Test 4 different depths
    'rf__n_estimators': [50, 100, 200]  # Test 3 different forest sizes
}

grid_search = GridSearchCV(
    estimator=final_pl,
    param_grid=param_grid,
    cv=5, # 5-Fold Cross validation
    scoring='accuracy'
)
grid_search.fit(X_train, y_train)

print(f"Best Hyperparameters Found: {grid_search.best_params_}")
print(f"Best 5-Fold CV Accuracy:    {grid_search.best_score_:.4f}")
print("-" * 40)

best_model = grid_search.best_estimator_
print(f"Final Test Accuracy:        {best_model.score(X_test, y_test):.4f}")

Best Hyperparameters Found: {'rf__max_depth': 5, 'rf__n_estimators': 200}
Best 5-Fold CV Accuracy:    0.7260
----------------------------------------
Final Test Accuracy:        0.7282


In [135]:
# 1. Extract the actual Random Forest model from inside your pipeline
rf_model = best_model.named_steps['rf']

# 2. Extract the importances (a list of percentages that add up to 1.0)
importances = rf_model.feature_importances_

# 3. Define the feature names in the exact order they went through the ColumnTransformer
# (StandardScaler did gold and kills first, passthrough did the other two last)
feature_names = ['golddiffat15', 'killsat15', 'firstblood', 'firstdragon']

# 4. Put them into a DataFrame and sort them
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=True)

# 5. Create a horizontal bar chart
fig_importance = px.bar(
    importance_df, 
    x='Importance', 
    y='Feature', 
    orientation='h', # Horizontal makes labels easier to read
    title='Random Forest Model: Feature Importances',
    labels={'Importance': 'Relative Importance (0 to 1.0)', 'Feature': ''},
    color_continuous_scale='Blues'
)

fig_importance.update_layout(width=900, height=400, showlegend=False)
fig_importance.show()
fig_importance.write_html('assets/importance_bar_chart.html', include_plotlyjs='cdn')

In [136]:
# Our predictions in action
df_scatter = X_test.copy()

# 2. Add the model's predictions as a new column
df_scatter['Predicted_Result'] = best_model.predict(X_test)

# 3. Map the 0s and 1s to text for a cleaner legend
df_scatter['Predicted_Result'] = df_scatter['Predicted_Result'].replace({0: 'Predicted Loss', 1: 'Predicted Win'})

# 4. Build the Scatter Plot
fig_scatter = px.scatter(
    df_scatter,
    x='golddiffat15',
    y='killsat15',
    color='Predicted_Result',
    color_discrete_map={'Predicted Win': '#2ca02c', 'Predicted Loss': '#d62728'},
    opacity=0.6, # Making dots slightly transparent helps when they overlap
    title="Decision Boundary: Model Predictions by Gold and Kills",
    labels={
        'golddiffat15': 'Gold Difference at 15 Minutes', 
        'killsat15': 'Total Kills at 15 Minutes'
    }
)

# Add a vertical line at 0 gold to mark "Dead Even"
fig_scatter.add_vline(x=0, line_dash="dash", line_color="black", opacity=0.5)

fig_scatter.update_layout(width=800, height=600)
fig_scatter.show()
fig_scatter.write_html('assets/scatter_predictions.html', include_plotlyjs='cdn')

## Step 8: Fairness Analysis

To evaluate the fairness of our Final Random Forest model, we utilized permutation testing to predict outcomes (win/loss) as accurately for teams that secure the first dragon as it does for teams that do not. We used accuracy parity as our evaluation metric to distinguish the difference in overal prediction accuracy for teams that do claim first dragon and for teams that don't. Our threshold of significance will be p-value > 0.05.

- Null Hypothesis: Our model is fair. Its accuracy between teams that get first dragon and teams that don't is the same.
- Alternative Hypothesis: Our model is unfair. Its accuracy between teams that get first dragon is significantly different from its accuracy for teams that don't.

Our observed difference in accuracy between the two groups was around 4.03%. After running the permutation test with 500 repititions, we calculated a p-value of 0.01120. Because our p-value of 0.01120 is greater than our threshold of significance (0.05), we fail to reject the null hypothesis. Therefore, we conclude that our model is fair and that there is no significant difference in predictive accuracy based on whether a team secures first dragon.

In [ ]:
# 1. Get the final predictions on the test set from your pipeline
y_pred = best_model.predict(X_test)

# 2. Create a DataFrame specifically for the Fairness Analysis
fairness_df = X_test.copy()
fairness_df['actual_result'] = y_test
fairness_df['predicted_result'] = y_pred

group_col = 'firstdragon'

# 3. Define a helper function to calculate the absolute difference in accuracy
def calc_abs_acc_diff(df, group_col):
    # Accuracy for Group X (Got First Dragon)
    group_x = df[df[group_col] == 1]
    acc_x = (group_x['actual_result'] == group_x['predicted_result']).mean()
    
    # Accuracy for Group Y (No First Dragon)
    group_y = df[df[group_col] == 0]
    acc_y = (group_y['actual_result'] == group_y['predicted_result']).mean()
    
    return abs(acc_x - acc_y)

# 4. Calculate the Observed Test Statistic (The real accuracy difference)
obs_diff = calc_abs_acc_diff(fairness_df, group_col)
print(f"Observed Accuracy Difference: {obs_diff:.4f}")

# 5. Run the Permutation Test
n_repetitions = 500
simulated_diffs = []
np.random.seed(42)
for _ in range(n_repetitions):
    # Shuffle the 'firstdragon' labels to simulate the Null Hypothesis
    shuffled_groups = np.random.permutation(fairness_df[group_col])
    
    # Assign the shuffled labels to a temporary dataframe
    temp_df = fairness_df.copy()
    temp_df[group_col] = shuffled_groups
    
    # Calculate the test statistic for this shuffled world
    simulated_diff = calc_abs_acc_diff(temp_df, group_col)
    simulated_diffs.append(simulated_diff)

# 6. Calculate the P-value
p_value = np.count_nonzero(np.array(simulated_diffs) >= obs_diff) / n_repetitions
print(f"Fairness P-Value: {p_value:.4f}")

Observed Accuracy Difference: 0.0403
Fairness P-Value: 0.1080


In [138]:
# 1. Put the simulated differences into a DataFrame for Plotly
sim_df = pd.DataFrame({'Simulated Accuracy Differences': simulated_diffs})

# 2. Plot the histogram
fig_fairness = px.histogram(
    sim_df, 
    x='Simulated Accuracy Differences',
    nbins=30,
    title='Fairness Permutation Test: Simulated vs. Observed Differences',
    color_discrete_sequence=['#1f77b4'] # Standard blue
)

# 3. Add a thick red vertical line for your Observed Difference
fig_fairness.add_vline(
    x=obs_diff, 
    line_dash='dash', 
    line_color='red', 
    line_width=3,
    annotation_text=f'Observed Diff: {obs_diff:.4f}', 
    annotation_position='top right'
)

fig_fairness.update_layout(width=800, height=500, yaxis_title="Frequency")
fig_fairness.show()

fig_scatter.write_html('assets/fairness_permutation_test_histogram.html', include_plotlyjs='cdn')